In [33]:
import pandas as pd
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("samarth1006/nthl-pool-v6",output_dir="./data")

print("Path to dataset files:", path)

/Users/tanmaynegi/codes/NameLLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: ./data


In [4]:
df = pd.read_parquet("./data/pool_v6.parquet")

In [5]:
df.head(5)

,prompt_id,text,model,src
0,v5:drjcfmd:n0408,Chaos theory explains how seemingly predictabl...,gemini-3.1-pro-preview,drjcfmd/ntl-generated-v2
1,v5:drjcfmd:n0173,"A prion is not a virus, not a bacterium, not e...",glm-5.3,drjcfmd/ntl-generated-v2
2,v5:pt:eb9452913f,"In the moments after birth, a newborn's body p...",kimi-k3,prvsiyan/prvsiyans-extended-name-that-llm-dataset
3,v5:pt:6951b70fb8,She does not panic; the silence inside her che...,gemini-3.8-flash,chosunghwan/ntl-gemini-new-prompts
4,v5:pt:9bb251e2f9,"Conceived by mathematician Tibor Radó in 1962,...",gemini-3.7-flash,chosunghwan/ntl-gemini-new-prompts


In [6]:
df.describe()

,prompt_id,text,model,src
count,57795,57795,57795,57795
unique,8145,57795,18,44
top,v5:dhruvtibarewal:,Chaos theory explains how seemingly predictabl...,claude-opus-4-8,drjcfmd/ntl-generated-v2
freq,680,1,3424,7836


In [7]:
df.shape

(57795, 4)

In [8]:
df = df.drop(columns = ["prompt_id","src"])

In [9]:
df

,text,model
0,Chaos theory explains how seemingly predictabl...,gemini-3.1-pro-preview
1,"A prion is not a virus, not a bacterium, not e...",glm-5.3
2,"In the moments after birth, a newborn's body p...",kimi-k3
3,She does not panic; the silence inside her che...,gemini-3.8-flash
4,"Conceived by mathematician Tibor Radó in 1962,...",gemini-3.7-flash
...,...,...
57790,Imagine a photon leaving a quasar before Earth...,gpt-5.6-terra
57791,Imagine your program has two storage rooms. Th...,kimi-k3
57792,"Beneath the shallow waters off Jamaica, the su...",gpt-5.6-sol
57793,"Under the midnight moon, the frozen lake gleam...",gpt-5.6-sol


In [11]:
sorted(df["model"].unique())

['claude-fable-5',
 'claude-opus-4-6',
 'claude-opus-4-8',
 'claude-opus-5',
 'claude-sonnet-5',
 'gemini-3.1-pro-preview',
 'gemini-3.6-flash',
 'gemini-3.7-flash',
 'gemini-3.8-flash',
 'glm-5.3',
 'glm-5.3-flash',
 'gpt-5.5',
 'gpt-5.6-luna',
 'gpt-5.6-sol',
 'gpt-5.6-terra',
 'gpt-6-astra',
 'kimi-k3',
 'qwen3.8-max']

In [22]:
def simplify_model(name): 
    if "claude" in name: 
        name = "claude"
    elif "gemini" in name: 
        name = "gemini"
    elif "glm" in name:
        name = "glm"
    elif "gpt" in name: 
        name = "chatgpt"
    elif "qwen" in name: 
        name = "qwen"
    else: 
        name = "kimi"
    return name


In [23]:
df['model'] = df['model'].apply(simplify_model)

In [24]:
df.head(5)

,text,model
0,Chaos theory explains how seemingly predictabl...,gemini
1,"A prion is not a virus, not a bacterium, not e...",glm
2,"In the moments after birth, a newborn's body p...",kimi
3,She does not panic; the silence inside her che...,gemini
4,"Conceived by mathematician Tibor Radó in 1962,...",gemini


In [25]:
df['model'].unique()

<ArrowStringArray>
['gemini', 'glm', 'kimi', 'claude', 'chatgpt', 'qwen']
Length: 6, dtype: str

STEP 1 : TFIDF VECTORIZATION


In [31]:
word_vec = TfidfVectorizer(stop_words=None, ngram_range=(1,3),min_df = 5,max_df = 0.95,max_features=25000,sublinear_tf= True)
char_vec = TfidfVectorizer(
    analyzer='char_wb',    # Character n-grams inside word boundaries
    ngram_range=(3, 5),
    min_df=5,
    max_features=25000,
    sublinear_tf=True
)

In [32]:
from sklearn.pipeline import FeatureUnion

union = FeatureUnion([
    ("word", word_vec),
    ("char", char_vec)
])

In [34]:
le = LabelEncoder()
y = le.fit_transform(df['model'])
X = df['text']

In [ ]:
X_train,X_test, y_train,t_test = train_test_split(X,y,stratify=y,random_state = 42, val)

array([2, 3, 4, 2, 2])